# Data Pipeline — Interactive Notebook

This notebook mirrors the data-processing pipeline from `code/data_cleaning/`. Each stage loads, cleans, and displays the resulting DataFrames so you can scroll through the data interactively.

## Setup

Load all required packages.

In [1]:
using DataFrames
using CSV
using Arrow
using Statistics
using Printf
using Plots
using HTTP, JSON3

### Paths

Adjust `PROJECT_ROOT` if your notebook lives somewhere other than `code/notebooks/`.

In [2]:
# ── Paths ──────────────────────────────────────────────────────
# Assumes this notebook is in:  project_root/code/notebooks/
const PROJECT_ROOT      = joinpath(@__DIR__, "..", "..")
const DATA_DIR          = joinpath(PROJECT_ROOT, "data")
const RAW_DIR           = joinpath(DATA_DIR, "raw")
const DERIVED_DIR       = joinpath(DATA_DIR, "derived")

const RAW_CPS_BASIC_DIR = joinpath(RAW_DIR, "cps_basic")
const RAW_CPS_ASEC_DIR  = joinpath(RAW_DIR, "cps_asec")
const RAW_JOLTS_DIR     = joinpath(RAW_DIR, "jolts")

mkpath(DERIVED_DIR)

println("PROJECT_ROOT = ", PROJECT_ROOT)
println("DERIVED_DIR  = ", DERIVED_DIR)

PROJECT_ROOT = /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/../..
DERIVED_DIR  = /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/../../data/derived


### Estimation Windows & Helper Functions

All the shared utilities from `utils.jl`.

In [3]:
# ============================================================
# Estimation windows
# ============================================================

const WINDOWS = Dict{Symbol, NamedTuple}(
    :base_fc     => (label = "Pre-FC baseline",
                     ym_start = (2003, 1),  ym_end = (2007, 12),
                     asec_years = 2004:2008),
    :crisis_fc   => (label = "Financial crisis",
                     ym_start = (2008, 1),  ym_end = (2009, 6),
                     asec_years = 2009:2010),
    :base_covid  => (label = "Pre-COVID baseline",
                     ym_start = (2015, 1),  ym_end = (2019, 12),
                     asec_years = 2016:2020),
    :crisis_covid => (label = "COVID crisis",
                      ym_start = (2020, 3),  ym_end = (2021, 12),
                      asec_years = 2021:2022),
)

function in_window(year::Int, month::Int, win::NamedTuple)::Bool
    ym = (year, month)
    return ym >= win.ym_start && ym <= win.ym_end
end

function in_asec_window(survey_year::Int, win::NamedTuple)::Bool
    return survey_year in win.asec_years
end

# ============================================================
# Classification helpers
# ============================================================

is_skilled(educ::Integer)::Bool = educ >= 111
in_age_range(age::Integer; lo::Int=16, hi::Int=64)::Bool = lo <= age <= hi
is_civilian_lf(labforce::Integer)::Bool = labforce == 2
is_employed(empstat::Integer)::Bool = empstat in (10, 12)
is_unemployed(empstat::Integer)::Bool = empstat in (20, 21, 22)
is_wage_worker(classwkr::Integer)::Bool = classwkr in (22, 25, 27, 28)
is_enrolled_no_ba(schlcoll::Integer, educ::Integer)::Bool = schlcoll in (4, 5) && educ < 111
valid_match_mish(mish::Integer)::Bool = mish in (1, 2, 3, 5, 6, 7)

# ============================================================
# Wage construction
# ============================================================

function compute_hourly_wage(incwage, wkswork1, uhrsworkly)::Float64
    (wkswork1 <= 0 || uhrsworkly <= 0 || incwage <= 0) && return NaN
    return Float64(incwage) / (Float64(wkswork1) * Float64(uhrsworkly))
end

deflate_wage(nominal_wage::Float64, cpi_t::Float64, cpi_base::Float64)::Float64 =
    nominal_wage * (cpi_base / cpi_t)

# ============================================================
# Trimming
# ============================================================

function winsorize_bounds(wages::Vector{Float64}, weights::Vector{Float64};
                          lo_pct::Float64=0.01, hi_pct::Float64=0.99)
    n = length(wages)
    n == 0 && return (NaN, NaN)
    idx = sortperm(wages)
    cum = 0.0
    total = sum(weights)
    lo_val = wages[idx[1]]
    hi_val = wages[idx[end]]
    for i in idx
        cum += weights[i] / total
        if cum >= lo_pct && lo_val == wages[idx[1]]
            lo_val = wages[i]
        end
        if cum >= hi_pct
            hi_val = wages[i]
            break
        end
    end
    return (lo_val, hi_val)
end

# ============================================================
# Weighted statistics
# ============================================================

function wmean(x::AbstractVector, w::AbstractVector)::Float64
    sw = sum(w); sw <= 0.0 && return NaN
    return sum(x .* w) / sw
end

function wmedian(x::AbstractVector, w::AbstractVector)::Float64
    n = length(x); n == 0 && return NaN
    idx = sortperm(x)
    cum = 0.0; total = sum(w)
    total <= 0.0 && return NaN
    for i in idx
        cum += w[i] / total
        cum >= 0.5 && return Float64(x[i])
    end
    return Float64(x[idx[end]])
end

function wvar(x::AbstractVector, w::AbstractVector)::Float64
    m = wmean(x, w); sw = sum(w)
    sw <= 0.0 && return NaN
    return sum(w .* (x .- m).^2) / sw
end

wsd(x::AbstractVector, w::AbstractVector)::Float64 = sqrt(max(wvar(x, w), 0.0))

function wcm3(x::AbstractVector, w::AbstractVector)::Float64
    m = wmean(x, w); sw = sum(w)
    sw <= 0.0 && return NaN
    return sum(w .* (x .- m).^3) / sw
end

# ============================================================
# COVID correction (placeholder)
# ============================================================

function apply_covid_correction(empstat::Integer, year::Int, month::Int)
    (year == 2020 && 3 <= month <= 6) || return empstat
    return empstat
end


# ============================================================
# JOLTS industry mapping
# ============================================================

function ind_to_jolts_supersector(ind::Int)::String
    170 <= ind <= 290   && return "AGRI"
    370 <= ind <= 490   && return "100000"
    570 <= ind <= 690   && return "480099"
    770 == ind          && return "230000"
    1070 <= ind <= 1990 && return "340000"
    2070 <= ind <= 3990 && return "320000"
    4070 <= ind <= 4590 && return "420000"
    4670 <= ind <= 5790 && return "440000"
    6070 <= ind <= 6390 && return "480099"
    6470 <= ind <= 6780 && return "510000"
    6870 <= ind <= 6990 && return "510099"
    7070 <= ind <= 7190 && return "510099"
    7270 <= ind <= 7790 && return "540099"
    7860 <= ind <= 7890 && return "610000"
    7970 <= ind <= 8470 && return "620000"
    8560 <= ind <= 8590 && return "710000"
    8660 <= ind <= 8690 && return "720000"
    8770 <= ind <= 9090 && return "810000"
    9170 <= ind <= 9590 && return "900000"
    return "UNKNOWN"
end

# ============================================================
# Moment name list
# ============================================================

const MOMENT_NAMES = [
    :ur_total, :ur_U, :ur_S, :skilled_share, :training_share,
    :emp_var_U, :emp_cm3_U, :emp_var_S, :emp_cm3_S,
    :jfr_U, :sep_rate_U, :jfr_S, :sep_rate_S,
    :ee_rate_S, :training_rate,
    :mean_wage_U, :mean_wage_S, :p50_wage_U, :p50_wage_S,
    :wage_premium, :wage_sd_U, :wage_sd_S,
    :theta_U, :theta_S,
]

println("All helpers loaded ✓")

All helpers loaded ✓


### Check Raw Data Directories

In [4]:
for (label, dir) in [("CPS Basic", RAW_CPS_BASIC_DIR),
                      ("CPS ASEC",  RAW_CPS_ASEC_DIR),
                      ("JOLTS",     RAW_JOLTS_DIR)]
    if isdir(dir)
        files = readdir(dir)
        @printf("  ✓ %-12s  %s  (%d files)\n", label, dir, length(files))
    else
        @printf("  ✗ %-12s  %s  (MISSING)\n", label, dir)
        mkpath(dir)
    end
end

  ✓ CPS Basic     /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/../../data/raw/cps_basic  (1 files)
  ✓ CPS ASEC      /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/../../data/raw/cps_asec  (2 files)
  ✓ JOLTS         /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/../../data/raw/jolts  (3 files)


---
## Stage 1 — Clean CPS Basic Monthly

Read raw IPUMS CPS Basic extract, restrict sample, classify skill / employment, apply COVID correction, assign windows, compute industry skill shares.

In [5]:
function clean_cps_basic()
    @info "clean_cps_basic: reading raw data..."

    raw_files = readdir(RAW_CPS_BASIC_DIR)
    csv_file  = filter(f -> endswith(f, ".csv") || endswith(f, ".csv.gz"), raw_files)
    arrow_file = filter(f -> endswith(f, ".arrow"), raw_files)

    if !isempty(arrow_file)
        df = DataFrame(Arrow.Table(joinpath(RAW_CPS_BASIC_DIR, first(arrow_file))))
    elseif !isempty(csv_file)
        df = CSV.read(joinpath(RAW_CPS_BASIC_DIR, first(csv_file)), DataFrame)
    else
        error("No CSV or Arrow file found in $(RAW_CPS_BASIC_DIR)")
    end
    @info "  Raw records: $(nrow(df))"

    rename!(df, [Symbol(uppercase(string(c))) => c for c in names(df)]...)

    filter!(row -> in_age_range(row.AGE), df)
    filter!(row -> is_civilian_lf(row.LABFORCE), df)
    @info "  After age/LF restriction: $(nrow(df))"

    df.skilled     = is_skilled.(df.EDUC)
    df.employed    = is_employed.(df.EMPSTAT)
    df.unemployed  = is_unemployed.(df.EMPSTAT)

    # COVID BLS correction
    for i in 1:nrow(df)
        corrected = apply_covid_correction(df.EMPSTAT[i], df.YEAR[i], df.MONTH[i])
        if corrected != df.EMPSTAT[i]
            df.employed[i]   = is_employed(corrected)
            df.unemployed[i] = is_unemployed(corrected)
        end
    end

    # Training flag
    if hasproperty(df, :SCHLCOLL)
        df.in_training = is_enrolled_no_ba.(df.SCHLCOLL, df.EDUC)
    else
        @warn "SCHLCOLL not in data — setting in_training = false"
        df.in_training = fill(false, nrow(df))
    end

    df.valid_match = valid_match_mish.(df.MISH)

    # Window assignment
    df.window = Vector{Symbol}(undef, nrow(df))
    fill!(df.window, :none)
    for i in 1:nrow(df)
        for (wname, wdef) in WINDOWS
            if in_window(df.YEAR[i], df.MONTH[i], wdef)
                df.window[i] = wname
                break
            end
        end
    end
    @info "  Observations in estimation windows: $(count(w -> w != :none, df.window)) / $(nrow(df))"

    # Industry skill shares
    if hasproperty(df, :IND)
        df.IND_JOLTS = ind_to_jolts_supersector.(df.IND)
        emp_df = filter(row -> row.employed && row.window != :none && row.IND_JOLTS != "AGRI" && row.IND_JOLTS != "UNKNOWN", df)
        ind_shares = combine(
            groupby(emp_df, [:window, :IND_JOLTS]),
            :skilled => mean => :skilled_share_ind,
            :WTFINL  => sum  => :weight_ind
        )
        Arrow.write(joinpath(DERIVED_DIR, "industry_skill_shares.arrow"), ind_shares)
        @info "  Industry skill shares saved ($(nrow(ind_shares)) rows)"
    end

    cols_to_keep = intersect(
        [:YEAR, :MONTH, :CPSID, :CPSIDP, :MISH, :WTFINL,
         :EMPSTAT, :EDUC, :AGE, :SEX, :IND,
         :skilled, :employed, :unemployed, :in_training,
         :valid_match, :window],
        Symbol.(names(df))
    )
    select!(df, cols_to_keep)

    outpath = joinpath(DERIVED_DIR, "cps_basic_clean.arrow")
    Arrow.write(outpath, df)
    @info "  Saved: $outpath  ($(nrow(df)) rows)"
    return df
end

cps_basic = clean_cps_basic()

┌ Info: clean_cps_basic: reading raw data...
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X13sZmlsZQ==.jl:2
┌ Info:   Raw records: 42310247
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X13sZmlsZQ==.jl:15
┌ Info:   After age/LF restriction: 20103840
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X13sZmlsZQ==.jl:21
┌ Info:   Observations in estimation windows: 11044410 / 20103840
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X13sZmlsZQ==.jl:57
┌ Info:   Industry skill shares saved (64 rows)
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_cell_df34fa98e697

Row,YEAR,MONTH,CPSID,CPSIDP,MISH,WTFINL,EMPSTAT,EDUC,AGE,SEX,IND,skilled,employed,unemployed,in_training,valid_match,window
,Int64,Int64,Int64,Int64,Int64,Float64?,Int64,Int64,Int64,Int64,Int64,Bool,Bool,Bool,Bool,Bool,Symbol
1,2000,1,19981003169300,19981003169301,8,453.883,10,111,38,1,50,true,true,false,false,false,none
2,2000,1,19981003169300,19981003169302,8,354.932,10,111,34,2,850,true,true,false,false,false,none
3,2000,1,19991004061800,19991004061801,8,373.839,10,73,32,1,42,false,true,false,false,false,none
4,2000,1,19991004061800,19991004061802,8,351.796,10,73,26,2,601,false,true,false,false,false,none
5,2000,1,19991004062100,19991004062101,8,321.778,10,91,54,1,50,false,true,false,false,false,none
6,2000,1,19981005364300,19981005364301,8,349.101,10,111,35,1,652,true,true,false,false,false,none
7,2000,1,19981004680000,19981004680001,8,348.367,10,92,31,2,633,false,true,false,false,false,none
8,2000,1,19981004680000,19981004680003,8,465.008,10,81,41,1,741,false,true,false,false,false,none
9,2000,1,19981204103000,19981204103001,6,233.378,12,73,31,2,60,false,true,false,false,true,none


### Explore CPS Basic

In [6]:
println("Size: $(nrow(cps_basic)) rows × $(ncol(cps_basic)) cols")
println("Columns: ", names(cps_basic))
first(cps_basic, 30)

Size: 20103840 rows × 17 cols
Columns: ["YEAR", "MONTH", "CPSID", "CPSIDP", "MISH", "WTFINL", "EMPSTAT", "EDUC", "AGE", "SEX", "IND", "skilled", "employed", "unemployed", "in_training", "valid_match", "window"]


Row,YEAR,MONTH,CPSID,CPSIDP,MISH,WTFINL,EMPSTAT,EDUC,AGE,SEX,IND,skilled,employed,unemployed,in_training,valid_match,window
,Int64,Int64,Int64,Int64,Int64,Float64?,Int64,Int64,Int64,Int64,Int64,Bool,Bool,Bool,Bool,Bool,Symbol
1,2000,1,19981003169300,19981003169301,8,453.883,10,111,38,1,50,true,true,false,false,false,none
2,2000,1,19981003169300,19981003169302,8,354.932,10,111,34,2,850,true,true,false,false,false,none
3,2000,1,19991004061800,19991004061801,8,373.839,10,73,32,1,42,false,true,false,false,false,none
4,2000,1,19991004061800,19991004061802,8,351.796,10,73,26,2,601,false,true,false,false,false,none
5,2000,1,19991004062100,19991004062101,8,321.778,10,91,54,1,50,false,true,false,false,false,none
6,2000,1,19981005364300,19981005364301,8,349.101,10,111,35,1,652,true,true,false,false,false,none
7,2000,1,19981004680000,19981004680001,8,348.367,10,92,31,2,633,false,true,false,false,false,none
8,2000,1,19981004680000,19981004680003,8,465.008,10,81,41,1,741,false,true,false,false,false,none
9,2000,1,19981204103000,19981204103001,6,233.378,12,73,31,2,60,false,true,false,false,true,none


In [7]:
describe(cps_basic)

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Union…,Any,Union…,Any,Int64,Type
1,YEAR,2011.12,2000,2011.0,2024,0,Int64
2,MONTH,6.15028,1,6.0,12,0,Int64
3,CPSID,2.01062e13,19981000005400,2.01009e13,20241206952700,0,Int64
4,CPSIDP,2.01062e13,19981000005401,2.01009e13,20241206952704,0,Int64
5,MISH,4.50234,1,4.0,8,0,Int64
6,WTFINL,2455.37,31.6835,2535.38,34716.2,2026407,"Union{Missing, Float64}"
7,EMPSTAT,10.6858,10,10.0,22,0,Int64
8,EDUC,87.6743,2,81.0,125,0,Int64
9,AGE,40.2807,16,41.0,64,0,Int64


In [8]:
# Distribution across estimation windows
combine(groupby(cps_basic, :window), nrow => :count)

Row,window,count
,Symbol,Int64
1,none,9059430
2,base_fc,4568290
3,crisis_fc,1388052
4,base_covid,3908094
5,crisis_covid,1179974


In [9]:
# Skilled vs unskilled counts
combine(groupby(cps_basic, :skilled), nrow => :count)

Row,skilled,count
,Bool,Int64
1,false,13488744
2,true,6615096


---
## Stage 2 — Clean CPS ASEC

Read raw IPUMS CPS ASEC extract, restrict to wage workers, construct real hourly wages, trim, normalise.

In [10]:
function clean_cps_asec()
    @info "clean_cps_asec: reading raw data..."

    raw_files  = readdir(RAW_CPS_ASEC_DIR)
    csv_file   = filter(f -> endswith(f, ".csv") || endswith(f, ".csv.gz"), raw_files)
    arrow_file = filter(f -> endswith(f, ".arrow"), raw_files)

    if !isempty(arrow_file)
        df = DataFrame(Arrow.Table(joinpath(RAW_CPS_ASEC_DIR, first(arrow_file))))
    elseif !isempty(csv_file)
        df = CSV.read(joinpath(RAW_CPS_ASEC_DIR, first(csv_file)), DataFrame)
    else
        error("No CSV or Arrow file found in $(RAW_CPS_ASEC_DIR)")
    end
    @info "  Raw records (full download): $(nrow(df))"

    rename!(df, [Symbol(uppercase(string(c))) => c for c in names(df)]...)

    # ── ASEC supplement filter + year restriction ────────────────────────────
    # Context: starting in 2022 IPUMS bundles both ASEC supplement respondents
    # (ASECFLAG == 1, have ASECWT / income variables) and March basic monthly
    # CPS respondents (ASECFLAG == 2, have WTFINL but no income variables) in
    # the same file.  For 2000–2021 IPUMS shipped pure ASEC files so this made
    # no difference; from 2022 the filter is essential.
    # Paper extraction window: ASEC survey years 2003–2022 (income years 2002–2021).
    n_raw = nrow(df)

    if hasproperty(df, :ASECFLAG)
        n_nonsupplement = count(row -> coalesce(row.ASECFLAG, 0) != 1, eachrow(df))
        filter!(row -> coalesce(row.ASECFLAG, 0) == 1, df)
        @info "  [ASEC filter] Non-supplement records dropped: $n_nonsupplement / $n_raw"
    else
        @warn "  ASECFLAG not in extract — skipping supplement filter."
    end

    n_after_flag = nrow(df)
    filter!(row -> 2003 <= row.YEAR <= 2022, df)
    n_out_of_window = n_after_flag - nrow(df)
    @info "  [Year filter]  Out-of-window records dropped: $n_out_of_window  (kept ASEC 2003–2022)"
    @info "  Records entering sample pipeline: $(nrow(df))"
    # ────────────────────────────────────────────────────────────────────────

    sort!(df, :YEAR)

    # ── Sample restrictions ──────────────────────────────────────────────────
    key_cols         = [:AGE, :CLASSWKR, :INCWAGE, :WKSWORK1, :UHRSWORKLY,
                        :ASECWT, :EDUC, :YEAR, :SEX]
    present_key_cols = intersect(key_cols, Symbol.(names(df)))
    n_before_drop    = nrow(df)
    n_dropped        = n_before_drop - nrow(dropmissing(df[!, present_key_cols]))
    @info "  Rows dropped for missingness in key cols: $n_dropped / $n_before_drop"
    dropmissing!(df, present_key_cols)

    filter!(row -> in_age_range(row.AGE), df)
    filter!(row -> is_wage_worker(row.CLASSWKR), df)
    filter!(row -> row.INCWAGE > 0 && row.WKSWORK1 > 0 && row.UHRSWORKLY > 0, df)
    @info "  After sample restrictions: $(nrow(df))"
    # ────────────────────────────────────────────────────────────────────────

    df.hourly_wage = compute_hourly_wage.(df.INCWAGE, df.WKSWORK1, df.UHRSWORKLY)
    filter!(row -> isfinite(row.hourly_wage) && row.hourly_wage > 0.0, df)

    # ── Deflate to constant (2012) dollars ───────────────────────────────────
    # ASEC survey year 2013 reports income year 2012 → use as CPI base.
    if hasproperty(df, :CPI99)
        idx_2013 = findfirst(==(2013), df.YEAR)
        cpi_2012 = isnothing(idx_2013) ? nothing : df.CPI99[idx_2013]
        if isnothing(cpi_2012) || ismissing(cpi_2012) || iszero(cpi_2012)
            cpi_2012 = 1.0
            @warn "  Could not identify CPI99 base for income year 2012; wages not deflated."
        end
        df.real_wage = deflate_wage.(df.hourly_wage, Float64.(df.CPI99), Float64(cpi_2012))
    else
        @warn "  CPI99 not in data — wages remain nominal."
        df.real_wage = df.hourly_wage
    end
    # ────────────────────────────────────────────────────────────────────────

    df.skilled = is_skilled.(df.EDUC)

    # ── Window assignment (ASEC survey year → income year) ───────────────────
    df.income_year = df.YEAR .- 1
    df.window      = Vector{Symbol}(undef, nrow(df))
    fill!(df.window, :none)
    for i in 1:nrow(df)
        for (wname, wdef) in WINDOWS
            if in_asec_window(df.YEAR[i], wdef)
                df.window[i] = wname
                break
            end
        end
    end
    @info "  Observations in estimation windows: $(count(w -> w != :none, df.window)) / $(nrow(df))"
    # ────────────────────────────────────────────────────────────────────────

    # ── Trimming: 1st/99th percentile within each window ────────────────────
    df.trimmed = fill(false, nrow(df))
    for wname in keys(WINDOWS)
        mask = df.window .== wname
        !any(mask) && continue
        wages_w   = df.real_wage[mask]
        weights_w = Float64.(df.ASECWT[mask])
        lo, hi    = winsorize_bounds(wages_w, weights_w)
        for i in findall(mask)
            if df.real_wage[i] < lo || df.real_wage[i] > hi
                df.trimmed[i] = true
            end
        end
    end
    @info "  Trimmed observations: $(count(df.trimmed))"
    filter!(row -> !row.trimmed, df)
    # ────────────────────────────────────────────────────────────────────────

    # ── Normalisation: divide by pooled median within each window ────────────
    df.wage_norm = fill(NaN, nrow(df))
    for wname in keys(WINDOWS)
        mask = df.window .== wname
        !any(mask) && continue
        wages_w   = df.real_wage[mask]
        weights_w = Float64.(df.ASECWT[mask])
        med_w     = wmedian(wages_w, weights_w)
        if isfinite(med_w) && med_w > 0.0
            df.wage_norm[mask] .= df.real_wage[mask] ./ med_w
        else
            @warn "  Median wage is invalid for window $wname"
            df.wage_norm[mask] .= df.real_wage[mask]
        end
    end
    # ────────────────────────────────────────────────────────────────────────

    cols_to_keep = intersect(
        [:YEAR, :income_year, :ASECWT, :AGE, :SEX, :EDUC,
         :skilled, :real_wage, :wage_norm, :window],
        Symbol.(names(df))
    )
    select!(df, cols_to_keep)

    outpath = joinpath(DERIVED_DIR, "cps_asec_clean.arrow")
    Arrow.write(outpath, df)
    @info "  Saved: $outpath  ($(nrow(df)) rows)"
    return df
end

cps_asec = clean_cps_asec()

┌ Info: clean_cps_asec: reading raw data...
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X23sZmlsZQ==.jl:2
┌ Info:   Raw records (full download): 9724563
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X23sZmlsZQ==.jl:15
┌ Info:   [ASEC filter] Non-supplement records dropped: 4829786 / 9724563
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X23sZmlsZQ==.jl:31
┌ Info:   [Year filter]  Out-of-window records dropped: 1001721  (kept ASEC 2003–2022)
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X23sZmlsZQ==.jl:39
┌ Info:   Records entering sample pipeline: 3893056
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Resea

Row,YEAR,income_year,ASECWT,AGE,SEX,EDUC,skilled,real_wage,wage_norm,window
,Int64,Int64,Float64,Int64,Int64,Int64,Bool,Float64,Float64,Symbol
1,2003,2002,297.04,47,1,73,false,19.6004,NaN,none
2,2003,2002,630.71,59,2,81,false,10.1771,NaN,none
3,2003,2002,532.97,44,2,73,false,5.72936,NaN,none
4,2003,2002,532.97,39,1,73,false,13.9465,NaN,none
5,2003,2002,310.77,42,1,73,false,13.5695,NaN,none
6,2003,2002,310.77,44,2,73,false,6.93554,NaN,none
7,2003,2002,657.5,35,2,111,true,11.7603,NaN,none
8,2003,2002,601.43,61,2,81,false,13.0663,NaN,none
9,2003,2002,288.76,32,1,111,true,16.9619,NaN,none


### Explore CPS ASEC

In [11]:
println("Size: $(nrow(cps_asec)) rows × $(ncol(cps_asec)) cols")
first(cps_asec, 30)

Size: 258677 rows × 10 cols


Row,YEAR,income_year,ASECWT,AGE,SEX,EDUC,skilled,real_wage,wage_norm,window
,Int64,Int64,Float64,Int64,Int64,Int64,Bool,Float64,Float64,Symbol
1,2003,2002,297.04,47,1,73,false,19.6004,NaN,none
2,2003,2002,630.71,59,2,81,false,10.1771,NaN,none
3,2003,2002,532.97,44,2,73,false,5.72936,NaN,none
4,2003,2002,532.97,39,1,73,false,13.9465,NaN,none
5,2003,2002,310.77,42,1,73,false,13.5695,NaN,none
6,2003,2002,310.77,44,2,73,false,6.93554,NaN,none
7,2003,2002,657.5,35,2,111,true,11.7603,NaN,none
8,2003,2002,601.43,61,2,81,false,13.0663,NaN,none
9,2003,2002,288.76,32,1,111,true,16.9619,NaN,none


In [12]:
describe(cps_asec)

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Union…,Any,Union…,Any,Int64,DataType
1,YEAR,2011.8,2003,2011.0,2022,0,Int64
2,income_year,2010.8,2002,2010.0,2021,0,Int64
3,ASECWT,1607.57,25.33,1466.07,31392.5,0,Float64
4,AGE,43.1187,16,44.0,64,0,Int64
5,SEX,1.58105,1,2.0,2,0,Int64
6,EDUC,98.1408,2,111.0,125,0,Int64
7,skilled,0.515388,false,1.0,true,0,Bool
8,real_wage,23.3546,0.00144231,19.742,3167.54,0,Float64
9,wage_norm,NaN,NaN,,NaN,0,Float64


In [13]:
# Wage stats by skill group
combine(groupby(cps_asec, :skilled),
    :real_wage => mean => :mean_real_wage,
    :wage_norm => mean => :mean_wage_norm,
    :real_wage => std  => :sd_real_wage,
    nrow => :count
)

Row,skilled,mean_real_wage,mean_wage_norm,sd_real_wage,count
,Bool,Float64,Float64,Float64,Int64
1,false,18.4079,NaN,13.9727,125358
2,true,28.0059,NaN,21.8277,133319


In [14]:
# Wage stats by window
combine(groupby(filter(r -> r.window != :none, cps_asec), :window),
    :real_wage => mean => :mean_real_wage,
    :wage_norm => mean => :mean_wage_norm,
    nrow => :count
)

Row,window,mean_real_wage,mean_wage_norm,count
,Symbol,Float64,Float64,Int64
1,base_fc,17.1452,1.10701,71559
2,crisis_fc,20.9993,1.11939,28570
3,base_covid,28.5605,1.13813,56385
4,crisis_covid,34.8954,1.1396,19814


In [15]:
# ── Diagnostic: CPS IND codes vs JOLTS supersectors ──────────
# What IND values does CPS actually have?
emp_in_window = filter(r -> r.employed && r.window != :none, cps_basic)

ind_summary = combine(
    groupby(emp_in_window, :IND),
    nrow => :count
)
sort!(ind_summary, :count; rev=true)

println("Unique IND codes in CPS (employed, in-window): $(nrow(ind_summary))")
println("\nTop 30 by frequency:")
display(first(ind_summary, 30))

println("\nAll unique IND codes (sorted numerically):")
display(sort(ind_summary.IND))

Row,IND,count
,Int64,Int64
1,770,775567
2,7860,650457
3,8680,582508
4,8190,417865
5,7870,260020
6,9470,198712
7,4970,173620
8,6990,167439
9,7380,161533


292-element Vector{Int64}:
  170
  180
  190
  270
  280
  290
  370
  380
  390
  470
    ⋮
 9290
 9370
 9380
 9390
 9470
 9480
 9490
 9570
 9590

Unique IND codes in CPS (employed, in-window): 292

Top 30 by frequency:

All unique IND codes (sorted numerically):


---
## Stage 3 — Clean JOLTS

Read BLS JOLTS flat files, parse series IDs, assign windows, merge with CPS industry skill shares, allocate vacancies to skill segments.

In [16]:
#ENV["BLS_API_KEY"] = "313cd5700eca43b4a20f72fa29bbb022"

In [17]:
const JOLTS_SERIES = [
    "JTS100000000000000JOL", "JTS230000000000000JOL",
    "JTS320000000000000JOL", "JTS340000000000000JOL",
    "JTS420000000000000JOL", "JTS440000000000000JOL",
    "JTS480099000000000JOL", "JTS510000000000000JOL",
    "JTS510099000000000JOL", "JTS540099000000000JOL",
    "JTS610000000000000JOL", "JTS620000000000000JOL",
    "JTS710000000000000JOL", "JTS720000000000000JOL",
    "JTS810000000000000JOL", "JTS910000000000000JOL",
    "JTS920000000000000JOL", "JTS000000000000000JOL",
    "JTS300000000000000JOL", "JTS400000000000000JOL",
    "JTS600000000000000JOL", "JTS700000000000000JOL",
    "JTS900000000000000JOL",
]

function _bls_fetch(series_ids, start_year, end_year)
    body = JSON3.write(Dict(
        "seriesid"        => series_ids,
        "startyear"       => string(start_year),
        "endyear"         => string(end_year),
        "registrationkey" => ENV["BLS_API_KEY"],
    ))
    resp = HTTP.post(
        "https://api.bls.gov/publicAPI/v2/timeseries/data/",
        ["Content-Type" => "application/json"],
        body,
    )
    return JSON3.read(String(resp.body))
end

function download_jolts()
    @info "download_jolts: fetching from BLS API..."

    records = @NamedTuple{series_id::String, year::Int, period::String, value::Float64}[]

    for (start_yr, end_yr) in [(2000, 2019), (2020, 2022)]
        @info "  Requesting $(start_yr)-$(end_yr)..."
        result = _bls_fetch(JOLTS_SERIES, start_yr, end_yr)

        result["status"] != "REQUEST_SUCCEEDED" &&
            error("BLS API error: $(result["message"])")

        for series in result["Results"]["series"]
            sid = String(series["seriesID"])
            for obs in series["data"]
                String(obs["period"]) == "M13" && continue
                push!(records, (
                    series_id = sid,
                    year      = parse(Int, String(obs["year"])),
                    period    = String(obs["period"]),
                    value     = parse(Float64, replace(String(obs["value"]), "," => "")),
                ))
            end
        end
        sleep(0.5)
    end

    df = DataFrame(records)

    missing_series = setdiff(JOLTS_SERIES, unique(df.series_id))
    if !isempty(missing_series)
        @warn "  No data returned for: $missing_series"
        @warn "  These series IDs may be incorrect — verify on bls.gov/jlt"
    end

    mkpath(RAW_JOLTS_DIR)
    outpath = joinpath(RAW_JOLTS_DIR, "jolts_openings.csv")
    CSV.write(outpath, df)
    @info "  Saved: $outpath  ($(nrow(df)) rows)"
    return df
end

download_jolts()

┌ Info: download_jolts: fetching from BLS API...
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y101sZmlsZQ==.jl:32
┌ Info:   Requesting 2000-2019...
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y101sZmlsZQ==.jl:37


KeyError: KeyError: key "BLS_API_KEY" not found

In [18]:
JOLTS_INDUSTRY_MAP = Dict(
    "100000" => "Mining and logging",
    "230000" => "Construction",
    "320000" => "Durable goods manufacturing",
    "340000" => "Nondurable goods manufacturing",
    "420000" => "Wholesale trade",
    "440000" => "Retail trade",
    "480099" => "Transportation, warehousing and utilities",
    "510000" => "Information",
    "510099" => "Financial activities",
    "540099" => "Professional and business services",
    "610000" => "Educational services",
    "620000" => "Health care and social assistance",
    "710000" => "Arts, entertainment, and recreation",
    "720000" => "Accommodation and food services",
    "810000" => "Other services",
    "910000" => "Federal government",
    "920000" => "State and local government",
    "000000" => "Total nonfarm",
    "300000" => "Manufacturing",
    "400000" => "Trade, transportation, and utilities",
    "600000" => "Education and health services",
    "700000" => "Leisure and hospitality",
    "900000" => "Government",
)

"""
    _parse_jolts_industry(sid) -> String

Pull the 6-digit industry code from a JOLTS series ID.
e.g. JTS320000000000000JOL -> "320000"
"""
function _parse_jolts_industry(sid::AbstractString)::String
    m = match(r"^JTS(\d{6})\d*JOL$", sid)
    isnothing(m) && return "unknown"
    return m.captures[1]
end

function clean_jolts()
    @info "clean_jolts: reading API-downloaded CSV..."

    # ── 1. Read the single long-format CSV produced by the Python script ──
    inpath = joinpath(RAW_JOLTS_DIR, "jolts_openings.csv")
    isfile(inpath) || error("File not found: $inpath — run the Python download script first.")

    df = CSV.read(inpath, DataFrame)
    rename!(df, [:series_id => :SERIES_ID, :year => :YEAR,
                 :period => :PERIOD, :value => :VALUE])
    @info "  Raw records: $(nrow(df))"

    # ── 2. Parse month from period string (M01 → 1, …, M12 → 12) ─────────
    df.MONTH = [parse(Int, replace(string(p), r"^M0?" => "")) for p in df.PERIOD]
    filter!(row -> 1 <= row.MONTH <= 12, df)

    # ── 3. Keep only JOL (job openings level) series ──────────────────────
    filter!(row -> endswith(string(row.SERIES_ID), "JOL"), df)

    # ── 4. Industry code + human-readable name ────────────────────────────
    df.industry_code = [_parse_jolts_industry(string(s)) for s in df.SERIES_ID]
    df.industry_name = [get(JOLTS_INDUSTRY_MAP, ic, "Unknown ($ic)")
                        for ic in df.industry_code]

    # Warn about any series that did not resolve
    unknown = filter(row -> row.industry_code == "unknown", df)
    if nrow(unknown) > 0
        @warn "  $(nrow(unknown)) records with unrecognised series ID — check regex"
        println(unique(unknown.SERIES_ID))
    end

    # ── 5. Assign estimation windows ──────────────────────────────────────
    df.window = Vector{Symbol}(undef, nrow(df))
    fill!(df.window, :none)
    for i in 1:nrow(df)
        for (wname, wdef) in WINDOWS
            if in_window(df.YEAR[i], df.MONTH[i], wdef)
                df.window[i] = wname
                break
            end
        end
    end

    # ── 6. Merge with CPS industry skill shares ───────────────────────────
    # Skill shares are keyed on (window, IND_JOLTS) where IND_JOLTS is the
    # 6-digit supersector code — must match industry_code here.
    # Cross-check aggregates (000000, 300000, …) are excluded from the join.
    shares_path = joinpath(DERIVED_DIR, "industry_skill_shares.arrow")
    if isfile(shares_path)
        shares = DataFrame(Arrow.Table(shares_path))
        @info "  Loaded industry skill shares ($(nrow(shares)) rows)"
        df = leftjoin(df, shares;
                      on = [:window => :window, :industry_code => :IND_JOLTS],
                      makeunique = true)
        for i in 1:nrow(df)
            if ismissing(df.skilled_share_ind[i])
                df.skilled_share_ind[i] = 0.5
            end
        end
    else
        @warn "Industry skill shares not found — using equal 50/50 split"
        df.skilled_share_ind = fill(0.5, nrow(df))
    end

    df.V_S = df.VALUE .* df.skilled_share_ind
    df.V_U = df.VALUE .* (1.0 .- df.skilled_share_ind)

    # ── 7. Aggregate over allocation series only (exclude cross-checks) ───
    # Cross-check aggregates share NAICS coverage with the sub-series,
    # so including them would double-count.
    CROSS_CHECK_CODES = Set(["000000","300000","400000","600000","700000","900000"])

    allocation = filter(row -> row.industry_code ∉ CROSS_CHECK_CODES &&
                               row.industry_code != "unknown", df)

    if nrow(allocation) == 0
        error("No allocation-series records found. Check that series IDs resolved correctly.")
    end

    agg = combine(
        groupby(allocation, [:YEAR, :MONTH, :window]),
        :V_S => sum => :V_S,
        :V_U => sum => :V_U,
    )

    outpath = joinpath(DERIVED_DIR, "jolts_clean.arrow")
    Arrow.write(outpath, agg)
    @info "  Saved: $outpath  ($(nrow(agg)) rows)"
    return agg
end

jolts = clean_jolts()

┌ Info: clean_jolts: reading API-downloaded CSV...
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X33sZmlsZQ==.jl:40
┌ Info:   Raw records: 6095
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X33sZmlsZQ==.jl:49
┌ Info:   Loaded industry skill shares (64 rows)
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X33sZmlsZQ==.jl:89
┌ Info:   Saved: /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/../../data/derived/jolts_clean.arrow  (265 rows)
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X33sZmlsZQ==.jl:126


Row,YEAR,MONTH,window,V_S,V_U
,Int64,Int64,Symbol,Float64,Float64
1,2019,12,base_covid,3764.44,8870.56
2,2019,11,base_covid,3870.38,9121.62
3,2019,10,base_covid,4068.28,9701.72
4,2019,9,base_covid,3975.78,9505.22
5,2019,8,base_covid,4019.79,9557.21
6,2019,7,base_covid,3939.66,9459.34
7,2019,6,base_covid,4026.82,9590.18
8,2019,5,base_covid,4063.71,9782.29
9,2019,4,base_covid,4028.05,9753.95


### Explore JOLTS

In [19]:
println("Size: $(nrow(jolts)) rows × $(ncol(jolts)) cols")
first(jolts, 30)

Size: 265 rows × 5 cols


Row,YEAR,MONTH,window,V_S,V_U
,Int64,Int64,Symbol,Float64,Float64
1,2019,12,base_covid,3764.44,8870.56
2,2019,11,base_covid,3870.38,9121.62
3,2019,10,base_covid,4068.28,9701.72
4,2019,9,base_covid,3975.78,9505.22
5,2019,8,base_covid,4019.79,9557.21
6,2019,7,base_covid,3939.66,9459.34
7,2019,6,base_covid,4026.82,9590.18
8,2019,5,base_covid,4063.71,9782.29
9,2019,4,base_covid,4028.05,9753.95


In [20]:
describe(jolts)

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Union…,Any,Union…,Any,Int64,DataType
1,YEAR,2011.46,2000,2011.0,2022,0,Int64
2,MONTH,6.52075,1,7.0,12,0,Int64
3,window,,base_covid,,none,0,Symbol
4,V_S,3487.89,1066.51,3307.81,11773.0,0,Float64
5,V_U,6238.71,2087.0,5907.7,14934.5,0,Float64


In [21]:
# Vacancies by window
combine(groupby(filter(r -> r.window != :none, jolts), :window),
    :V_S => mean => :mean_V_S,
    :V_U => mean => :mean_V_U,
    nrow => :n_months
)

Row,window,mean_V_S,mean_V_U,n_months
,Symbol,Float64,Float64,Int64
1,base_covid,3583.35,8528.22,60
2,crisis_fc,1552.41,4836.86,18
3,base_fc,1783.9,5918.32,60
4,crisis_covid,5006.84,10692.7,22


---
## Stage 4 — Build Transition Rates

Match persons across adjacent CPS months using CPSIDP, compute job-finding and separation rates by skill group.

In [22]:
function make_transitions()
    @info "make_transitions: loading cleaned CPS Basic..."

    inpath = joinpath(DERIVED_DIR, "cps_basic_clean.arrow")
    !isfile(inpath) && error("cps_basic_clean.arrow not found — run Stage 1 first")
    df = DataFrame(Arrow.Table(inpath))

    @info "  Building matched month-pairs..."
    matchable = filter(row -> row.valid_match && row.CPSIDP > 0, df)
    next_ym(y, m) = m == 12 ? (y + 1, 1) : (y, m + 1)

    lookup = Dict{Tuple{Int64, Int, Int}, NamedTuple}()
    for row in eachrow(df)
        key = (Int64(row.CPSIDP), row.YEAR, row.MONTH)
        lookup[key] = (skilled=row.skilled, employed=row.employed,
                       unemployed=row.unemployed, weight=row.WTFINL,
                       window=row.window)
    end

    pairs_data = NamedTuple[]
    for row in eachrow(matchable)
        ny, nm = next_ym(row.YEAR, row.MONTH)
        next_key = (Int64(row.CPSIDP), ny, nm)
        haskey(lookup, next_key) || continue
        next = lookup[next_key]
        push!(pairs_data, (
            year_t=row.YEAR, month_t=row.MONTH, skilled=row.skilled,
            emp_t=row.employed, unemp_t=row.unemployed,
            emp_t1=next.employed, unemp_t1=next.unemployed,
            weight=Float64(coalesce(row.WTFINL, 0.0)), window=row.window,
        ))
    end
    pairs = DataFrame(pairs_data)
    @info "  Matched pairs: $(nrow(pairs))"

    # Monthly transition rates
    results = NamedTuple[]
    for gk in groupby(pairs, [:year_t, :month_t, :skilled, :window])
        g = DataFrame(gk)
        sk = g.skilled[1]; win = g.window[1]
        u_mask  = g.unemp_t
        ue_mask = g.unemp_t .& g.emp_t1
        denom_jfr = sum(g.weight[u_mask])
        numer_jfr = sum(g.weight[ue_mask])
        jfr = denom_jfr > 0 ? numer_jfr / denom_jfr : NaN
        e_mask  = g.emp_t
        eu_mask = g.emp_t .& g.unemp_t1
        denom_sep = sum(g.weight[e_mask])
        numer_sep = sum(g.weight[eu_mask])
        sep = denom_sep > 0 ? numer_sep / denom_sep : NaN
        push!(results, (year=g.year_t[1], month=g.month_t[1], skilled=sk,
                         window=win, jfr=jfr, sep=sep, n_pairs=nrow(g)))
    end
    rates = DataFrame(results)

    # Window averages
    window_rates = NamedTuple[]
    for gk in groupby(filter(r -> r.window != :none, rates), [:window, :skilled])
        g = DataFrame(gk)
        valid_jfr = filter(isfinite, g.jfr)
        valid_sep = filter(isfinite, g.sep)
        push!(window_rates, (
            window=g.window[1], skilled=g.skilled[1],
            mean_jfr=isempty(valid_jfr) ? NaN : mean(valid_jfr),
            mean_sep=isempty(valid_sep) ? NaN : mean(valid_sep),
            n_months=nrow(g),
        ))
    end
    agg = DataFrame(window_rates)

    Arrow.write(joinpath(DERIVED_DIR, "transitions_monthly.arrow"), rates)
    outpath = joinpath(DERIVED_DIR, "transitions.arrow")
    Arrow.write(outpath, agg)
    @info "  Saved: $outpath  ($(nrow(agg)) rows)"
    return agg
end

transitions = make_transitions()

┌ Info: make_transitions: loading cleaned CPS Basic...
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X42sZmlsZQ==.jl:2
┌ Info:   Building matched month-pairs...
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X42sZmlsZQ==.jl:8
┌ Info:   Matched pairs: 13129067
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X42sZmlsZQ==.jl:34
┌ Info:   Saved: /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/../../data/derived/transitions.arrow  (8 rows)
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X42sZmlsZQ==.jl:74


Row,window,skilled,mean_jfr,mean_sep,n_months
,Symbol,Bool,Float64,Float64,Int64
1,base_fc,true,0.330699,0.00632283,60
2,base_fc,false,0.34774,0.0156321,60
3,crisis_fc,false,0.280844,0.0202718,18
4,crisis_fc,true,0.277446,0.00781182,18
5,base_covid,false,0.349423,0.0136774,60
6,base_covid,true,0.332317,0.00600562,60
7,crisis_covid,true,0.314021,0.01154,22
8,crisis_covid,false,0.344064,0.0247488,22


### Explore Transitions

In [23]:
println("Window-averaged transition rates:")
transitions

Window-averaged transition rates:


Row,window,skilled,mean_jfr,mean_sep,n_months
,Symbol,Bool,Float64,Float64,Int64
1,base_fc,true,0.330699,0.00632283,60
2,base_fc,false,0.34774,0.0156321,60
3,crisis_fc,false,0.280844,0.0202718,18
4,crisis_fc,true,0.277446,0.00781182,18
5,base_covid,false,0.349423,0.0136774,60
6,base_covid,true,0.332317,0.00600562,60
7,crisis_covid,true,0.314021,0.01154,22
8,crisis_covid,false,0.344064,0.0247488,22


In [24]:
# Also look at monthly rates
rates_monthly = DataFrame(Arrow.Table(joinpath(DERIVED_DIR, "transitions_monthly.arrow")))
println("Monthly rates: $(nrow(rates_monthly)) rows")
first(rates_monthly, 30)

Monthly rates: 598 rows


Row,year,month,skilled,window,jfr,sep,n_pairs
,Int64,Int64,Bool,Symbol,Float64,Float64,Int64
1,2000,1,false,none,0.382721,0.0138429,30416
2,2000,1,true,none,0.463389,0.00476413,11912
3,2000,2,false,none,0.45256,0.0140722,29509
4,2000,2,true,none,0.448672,0.00416224,11601
5,2000,3,false,none,0.448989,0.0115888,30129
6,2000,3,true,none,0.363739,0.00459542,11451
7,2000,4,false,none,0.423707,0.0133853,30517
8,2000,4,true,none,0.362997,0.00485278,11484
9,2000,5,false,none,0.483633,0.015393,30038


In [25]:
describe(rates_monthly)

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Union…,Any,Union…,Any,Int64,DataType
1,year,2011.96,2000,2012.0,2024,0,Int64
2,month,6.48161,1,6.0,12,0,Int64
3,skilled,0.5,false,0.5,true,0,Bool
4,window,,base_covid,,none,0,Symbol
5,jfr,0.319412,0.159705,0.32111,0.536468,0,Float64
6,sep,0.0118288,0.00318667,0.0110003,0.148286,0,Float64
7,n_pairs,21955.0,10811,17324.0,68228,0,Int64


---
## Stage 5 — Compute All Moments

24 targeted moments × 4 estimation windows.

In [26]:
function _load_arrow(filename::String)::DataFrame
    path = joinpath(DERIVED_DIR, filename)
    !isfile(path) && (@warn "$filename not found"; return DataFrame())
    return DataFrame(Arrow.Table(path))
end

function _compute_stock_moments(cps_w::DataFrame)::Dict{Symbol, Float64}
    moments = Dict{Symbol, Float64}()
    monthly = NamedTuple[]
    for gk in groupby(cps_w, [:YEAR, :MONTH])
        g = DataFrame(gk)
        w = Float64.(coalesce.(g.WTFINL, 0.0)); sw = sum(w)
        sw <= 0 && continue
        n_unemp   = sum(w[g.unemployed])
        n_unemp_U = sum(w[g.unemployed .& .!g.skilled])
        n_unemp_S = sum(w[g.unemployed .& g.skilled])
        n_lf = sw; n_lf_U = sum(w[.!g.skilled]); n_lf_S = sum(w[g.skilled])
        ur_total = n_lf > 0 ? n_unemp / n_lf : NaN
        ur_U = n_lf_U > 0 ? n_unemp_U / n_lf_U : NaN
        ur_S = n_lf_S > 0 ? n_unemp_S / n_lf_S : NaN
        skilled_share = n_lf > 0 ? n_lf_S / n_lf : NaN
        training_share = hasproperty(g, :in_training) ? (n_lf > 0 ? sum(w[g.in_training]) / n_lf : NaN) : NaN
        push!(monthly, (ur_total=ur_total, ur_U=ur_U, ur_S=ur_S,
                         skilled_share=skilled_share, training_share=training_share))
    end
    if !isempty(monthly)
        mdf = DataFrame(monthly)
        moments[:ur_total]       = mean(filter(isfinite, mdf.ur_total))
        moments[:ur_U]           = mean(filter(isfinite, mdf.ur_U))
        moments[:ur_S]           = mean(filter(isfinite, mdf.ur_S))
        moments[:skilled_share]  = mean(filter(isfinite, mdf.skilled_share))
        moments[:training_share] = mean(filter(isfinite, mdf.training_share))
    end
    return moments
end

function _fill_transition_moments!(moments::Dict{Symbol, Float64}, trans_w::DataFrame)
    for row in eachrow(trans_w)
        if row.skilled
            moments[:jfr_S]      = row.mean_jfr
            moments[:sep_rate_S] = row.mean_sep
        else
            moments[:jfr_U]      = row.mean_jfr
            moments[:sep_rate_U] = row.mean_sep
        end
    end
    moments[:ee_rate_S] = get(moments, :ee_rate_S, NaN)
end

function _compute_wage_moments(asec_w::DataFrame)::Dict{Symbol, Float64}
    moments = Dict{Symbol, Float64}()
    unskilled = filter(row -> !row.skilled, asec_w)
    skilled   = filter(row ->  row.skilled, asec_w)
    if nrow(unskilled) > 0
        wu = Float64.(unskilled.wage_norm); wt = Float64.(unskilled.ASECWT)
        moments[:mean_wage_U] = wmean(wu, wt); moments[:p50_wage_U] = wmedian(wu, wt)
        moments[:wage_sd_U] = wsd(wu, wt); moments[:emp_var_U] = wvar(wu, wt)
        moments[:emp_cm3_U] = wcm3(wu, wt)
    end
    if nrow(skilled) > 0
        ws = Float64.(skilled.wage_norm); wt = Float64.(skilled.ASECWT)
        moments[:mean_wage_S] = wmean(ws, wt); moments[:p50_wage_S] = wmedian(ws, wt)
        moments[:wage_sd_S] = wsd(ws, wt); moments[:emp_var_S] = wvar(ws, wt)
        moments[:emp_cm3_S] = wcm3(ws, wt)
    end
    if nrow(unskilled) > 0 && nrow(skilled) > 0
        log_wu = log.(max.(Float64.(unskilled.wage_norm), 1e-14))
        log_ws = log.(max.(Float64.(skilled.wage_norm), 1e-14))
        moments[:wage_premium] = wmean(log_ws, Float64.(skilled.ASECWT)) - wmean(log_wu, Float64.(unskilled.ASECWT))
    end
    return moments
end

function _compute_tightness(jolts_w::DataFrame, cps_w::DataFrame)::Dict{Symbol, Float64}
    moments = Dict{Symbol, Float64}()
    mean_V_U = mean(jolts_w.V_U); mean_V_S = mean(jolts_w.V_S)
    monthly_U = NamedTuple[]
    for gk in groupby(cps_w, [:YEAR, :MONTH])
        g = DataFrame(gk); w = Float64.(coalesce.(g.WTFINL, 0.0))
        push!(monthly_U, (U_U=sum(w[g.unemployed .& .!g.skilled]),
                           U_S=sum(w[g.unemployed .& g.skilled])))
    end
    mdf = DataFrame(monthly_U)
    mean_U_U = mean(mdf.U_U); mean_U_S = mean(mdf.U_S)
    moments[:theta_U] = mean_U_U > 0 ? mean_V_U / mean_U_U : NaN
    moments[:theta_S] = mean_U_S > 0 ? mean_V_S / mean_U_S : NaN
    return moments
end

function make_moments()
    @info "make_moments: assembling all 24 moments × 4 windows..."
    cps_basic_m = _load_arrow("cps_basic_clean.arrow")
    cps_asec_m  = _load_arrow("cps_asec_clean.arrow")
    trans       = _load_arrow("transitions.arrow")
    jolts_m     = _load_arrow("jolts_clean.arrow")

    all_moments = Dict{Symbol, DataFrame}()
    for (wname, wdef) in WINDOWS
        @info "  Window: $(wdef.label) ($wname)"
        moments = Dict{Symbol, Float64}()
        cps_w = filter(row -> row.window == wname, cps_basic_m)
        nrow(cps_w) > 0 ? merge!(moments, _compute_stock_moments(cps_w)) :
            (for k in [:ur_total,:ur_U,:ur_S,:skilled_share,:training_share]; moments[k]=NaN; end)
        trans_w = filter(row -> row.window == wname, trans)
        nrow(trans_w) > 0 ? _fill_transition_moments!(moments, trans_w) :
            (for k in [:jfr_U,:sep_rate_U,:jfr_S,:sep_rate_S,:ee_rate_S,:training_rate]; moments[k]=NaN; end)
        if haskey(moments, :training_share) && haskey(moments, :ur_U)
            ts = moments[:training_share]; ur = moments[:ur_U]
            moments[:training_rate] = (ts+ur) > 0 ? ts/(ts+ur) : NaN
        end
        asec_w = filter(row -> row.window == wname, cps_asec_m)
        nrow(asec_w) > 0 ? merge!(moments, _compute_wage_moments(asec_w)) :
            (for k in [:mean_wage_U,:mean_wage_S,:p50_wage_U,:p50_wage_S,:wage_premium,:wage_sd_U,:wage_sd_S,:emp_var_U,:emp_cm3_U,:emp_var_S,:emp_cm3_S]; moments[k]=NaN; end)
        jolts_w = filter(row -> row.window == wname, jolts_m)
        (nrow(jolts_w) > 0 && nrow(cps_w) > 0) ? merge!(moments, _compute_tightness(jolts_w, cps_w)) :
            (moments[:theta_U]=NaN; moments[:theta_S]=NaN)

        moment_df = DataFrame(moment=String[], value=Float64[], se=Float64[], weight=Float64[])
        for mname in MOMENT_NAMES
            val = get(moments, mname, NaN)
            se = isfinite(val) && val != 0.0 ? abs(val)*0.10 : 1.0
            wt = isfinite(se) && se > 0.0 ? 1.0/se^2 : 0.0
            push!(moment_df, (string(mname), val, se, wt))
        end
        CSV.write(joinpath(DERIVED_DIR, "moments_$(wname).csv"), moment_df)

        K = length(MOMENT_NAMES)
        sigma = zeros(K, K)
        for (i, _) in enumerate(MOMENT_NAMES); sigma[i,i] = moment_df.se[i]^2; end
        sigma_df = DataFrame(sigma, [string(m) for m in MOMENT_NAMES])
        CSV.write(joinpath(DERIVED_DIR, "sigma_$(wname).csv"), sigma_df)

        all_moments[wname] = moment_df
    end
    @info "  All moment files saved to $(DERIVED_DIR)"
    return all_moments
end

all_moments = make_moments()

┌ Info: make_moments: assembling all 24 moments × 4 windows...
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X51sZmlsZQ==.jl:91
┌ Info:   Window: COVID crisis (crisis_covid)
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X51sZmlsZQ==.jl:99
┌ Info:   Window: Pre-FC baseline (base_fc)
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X51sZmlsZQ==.jl:99
┌ Info:   Window: Financial crisis (crisis_fc)
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X51sZmlsZQ==.jl:99
┌ Info:   Window: Pre-COVID baseline (base_covid)
└ @ Main /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/jl_notebook_ce

Dict{Symbol, DataFrame} with 4 entries:
  :crisis_covid => 24×4 DataFrame…
  :base_fc      => 24×4 DataFrame…
  :crisis_fc    => 24×4 DataFrame…
  :base_covid   => 24×4 DataFrame…

### Explore Moments

In [27]:
# Print all moments for each window
for (wname, mdf) in all_moments
    println("\n═══ Window: $wname ═══")
    display(mdf)
end

Row,moment,value,se,weight
,String,Float64,Float64,Float64
1,ur_total,0.0706409,0.00706409,20039.5
2,ur_U,0.0885845,0.00885845,12743.4
3,ur_S,0.0432638,0.00432638,53425.9
4,skilled_share,0.394808,0.0394808,641.546
5,training_share,0.452943,0.0452943,487.43
6,emp_var_U,0.284857,0.0284857,1232.38
7,emp_cm3_U,0.30728,0.030728,1059.08
8,emp_var_S,0.432129,0.0432129,535.516
9,emp_cm3_S,0.371476,0.0371476,724.666


Row,moment,value,se,weight
,String,Float64,Float64,Float64
1,ur_total,0.0528946,0.00528946,35741.9
2,ur_U,0.0639419,0.00639419,24458.4
3,ur_S,0.0260133,0.00260133,1.47778e5
4,skilled_share,0.291534,0.0291534,1176.58
5,training_share,0.0942795,0.00942795,11250.3
6,emp_var_U,0.259064,0.0259064,1490.0
7,emp_cm3_U,0.17793,0.017793,3158.66
8,emp_var_S,0.403991,0.0403991,612.712
9,emp_cm3_S,0.244887,0.0244887,1667.51


Row,moment,value,se,weight
,String,Float64,Float64,Float64
1,ur_total,0.070038,0.0070038,20386.1
2,ur_U,0.0862959,0.00862959,13428.3
3,ur_S,0.0333158,0.00333158,90094.5
4,skilled_share,0.30682,0.030682,1062.26
5,training_share,0.0869284,0.00869284,13233.6
6,emp_var_U,0.246957,0.0246957,1639.67
7,emp_cm3_U,0.16724,0.016724,3575.35
8,emp_var_S,0.411193,0.0411193,591.438
9,emp_cm3_S,0.273733,0.0273733,1334.59


Row,moment,value,se,weight
,String,Float64,Float64,Float64
1,ur_total,0.0451597,0.00451597,49034.0
2,ur_U,0.0565659,0.00565659,31252.9
3,ur_S,0.0246717,0.00246717,1.64286e5
4,skilled_share,0.359648,0.0359648,773.115
5,training_share,0.482394,0.0482394,429.73
6,emp_var_U,0.289143,0.0289143,1196.12
7,emp_cm3_U,0.266533,0.0266533,1407.66
8,emp_var_S,0.457498,0.0457498,477.774
9,emp_cm3_S,0.363088,0.0363088,758.535



═══ Window: crisis_covid ═══

═══ Window: base_fc ═══

═══ Window: crisis_fc ═══

═══ Window: base_covid ═══


In [28]:
# Quick comparison: pick two windows side by side
if haskey(all_moments, :base_fc) && haskey(all_moments, :crisis_fc)
    comparison = DataFrame(
        moment = all_moments[:base_fc].moment,
        base_fc = all_moments[:base_fc].value,
        crisis_fc = all_moments[:crisis_fc].value,
    )
    comparison.diff = comparison.crisis_fc .- comparison.base_fc
    comparison
end

Row,moment,base_fc,crisis_fc,diff
,String,Float64,Float64,Float64
1,ur_total,0.0528946,0.070038,0.0171434
2,ur_U,0.0639419,0.0862959,0.022354
3,ur_S,0.0260133,0.0333158,0.00730252
4,skilled_share,0.291534,0.30682,0.0152862
5,training_share,0.0942795,0.0869284,-0.00735108
6,emp_var_U,0.259064,0.246957,-0.0121069
7,emp_cm3_U,0.17793,0.16724,-0.0106895
8,emp_var_S,0.403991,0.411193,0.00720149
9,emp_cm3_S,0.244887,0.273733,0.0288452


---
## Pipeline Summary

List all derived files produced by the pipeline.

In [29]:
println("Derived files in: $DERIVED_DIR")
for f in sort(readdir(DERIVED_DIR))
    sz = filesize(joinpath(DERIVED_DIR, f))
    @printf("  %-40s  %s\n", f, sz < 1024 ? "$sz B" : sz < 1024^2 ? "$(round(sz/1024; digits=1)) KB" : "$(round(sz/1024^2; digits=1)) MB")
end

Derived files in: /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/../../data/derived
  cps_asec_clean.arrow                      18.6 MB
  cps_basic_clean.arrow                     1906.0 MB
  industry_skill_shares.arrow               3.2 KB
  jolts_clean.arrow                         12.5 KB
  moments_base_covid.csv                    1.6 KB
  moments_base_fc.csv                       1.6 KB
  moments_crisis_covid.csv                  1.6 KB
  moments_crisis_fc.csv                     1.6 KB
  sigma_base_covid.csv                      2.9 KB
  sigma_base_fc.csv                         2.9 KB
  sigma_crisis_covid.csv                    2.9 KB
  sigma_crisis_fc.csv                       2.9 KB
  transitions.arrow                         1.7 KB
  transitions_monthly.arrow                 31.3 KB
